In [48]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import date

load_dotenv(dotenv_path='../.env')
TOKEN = os.getenv('FINMIND_TOKEN')
BASE_URL = 'https://api.finmindtrade.com/api/v4/data'

print('Token 載入成功！' if TOKEN else '⚠️ Token 載入失敗，請檢查 .env')

# pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

Token 載入成功！


### 1. 股價資料探索（TaiwanStockPrice）

In [21]:
def fetch_stock_data(stock_id,  begin_date= date.today().strftime('%Y-%m-%d'), end_date= date.today().strftime('%Y-%m-%d')):
    params = {
    'dataset': 'TaiwanStockPrice',
    'data_id': stock_id,
    'start_date': begin_date,
    'end_date': end_date,
    'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

"""
    [TaiwanStockPrice][schema]
    date: str, # 日期 (start from 1994-10-01)
    stock_id: str, # 股票代碼
    Trading_Volume: int64, # 成交量
    Trading_money: int64, # 成交金額
    open: float64, # 開盤價
    max: float64, # 最高價
    min: float64, # 最低價
    close: float64, # 收盤價
    spread: float64, # 漲跌幅
    Trading_turnover: float32 # 交易筆數
}"""

'\n    [TaiwanStockPrice][schema]\n    date: str, # 日期\n    stock_id: str, # 股票代碼\n    Trading_Volume: int64, # 成交量\n    Trading_money: int64, # 成交金額\n    open: float64, # 開盤價\n    max: float64, # 最高價\n    min: float64, # 最低價\n    close: float64, # 收盤價\n    spread: float64, # 漲跌幅\n    Trading_turnover: float32 # 交易筆數\n}'

In [18]:
# 非付費會員只能一次拿一支股票
tmp = fetch_stock_data('2330',"2021-01-01","2024-01-01")
tmp.head(5)

,date,stock_id,Trading_Volume,Trading_money,open,max,min,close,spread,Trading_turnover
0,2021-01-04,2330,39489959,21127581248,530.0,540.0,528.0,536.0,6.0,33316
1,2021-01-05,2330,34839391,18761831567,536.0,542.0,535.0,542.0,6.0,28512
2,2021-01-06,2330,55614434,30572783229,555.0,555.0,541.0,549.0,7.0,55462
3,2021-01-07,2330,53392763,30018630685,554.0,570.0,553.0,565.0,16.0,47905
4,2021-01-08,2330,62957148,36339702855,580.0,580.0,571.0,580.0,15.0,56426


### 2. 台股總覽探索 (TaiwanStockInfo) 

In [24]:
def fetch_stock_info():
    params = {
    'dataset': 'TaiwanStockInfo',
    'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

"""
    [TaiwanStockInfo][schema]
    industry_category: str, # 產業別
    stock_id: str, # 股票代碼
    stock_name: str, # 股票名稱
    type: str, # 市場別
    date: str # 更新日期
"""

'\n    [TaiwanStockInfo][schema]\n    industry_category: str, # 產業別\n    stock_id: str, # 股票代碼\n    stock_name: str, # 股票名稱\n    type: str, # 市場別\n    date: str # 更新日期\n'

In [30]:
tmp = fetch_stock_info()
tmp.loc[tmp['stock_name'].str.contains('聯合')]

,industry_category,stock_id,stock_name,type,date
108,電子零組件業,8080,永利聯合,tpex,2022-06-03
284,生技醫療業,4129A,聯合甲特,tpex,2023-08-10
2176,半導體業,6927,聯合聚晶,emerging,2026-03-05
3157,光電業,3576,聯合再生,twse,2026-03-05
3158,電子工業,3576,聯合再生,twse,2026-03-05
3515,生技醫療業,4129,聯合,tpex,2026-03-05


### 3. 財報資料探索 (TaiwanStockFinancialStatements)

In [55]:
# 損益表
def fetch_stock_financial_statements(stock_id,  begin_date= date.today().strftime('%Y-%m-%d'), end_date= date.today().strftime('%Y-%m-%d')):
    params = {
        'dataset': 'TaiwanStockFinancialStatements',
        'data_id': stock_id,
        'start_date': begin_date,
        'end_date': end_date,
        'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

"""
    [TaiwanStockFinancialStatements][schema]
    date: str, # 日期
    stock_id: str, # 股票代碼
    type: str, # 類別
    value: float64, # 數值
    origin_name: str # 原始名稱
"""


'\n    [TaiwanStockFinancialStatements][schema]\n    date: str, # 日期\n    stock_id: str, # 股票代碼\n    type: str, # 類別\n    value: float64, # 數值\n    origin_name: str # 原始名稱\n'

In [58]:
tmp = fetch_stock_financial_statements("2330","2025-12-01")
tmp

,date,stock_id,type,value,origin_name
0,2025-12-31,2330,OTHNOE,"1,106,367,000.0",其他收益及費損淨額
1,2025-12-31,2330,OtherComprehensiveIncome,"71,076,498,000.0",其他綜合損益（淨額）
2,2025-12-31,2330,TAX,"86,947,868,000.0",所得稅費用（利益）
3,2025-12-31,2330,IncomeAfterTaxes,"505,415,333,000.0",本期淨利（淨損）
4,2025-12-31,2330,TotalConsolidatedProfitForThePeriod,"576,491,831,000.0",本期綜合損益總額
5,2025-12-31,2330,EquityAttributableToOwnersOfParent,"505,743,990,000.0",淨利（淨損）歸屬於母公司業主
6,2025-12-31,2330,NoncontrollingInterests,"-328,657,000.0",淨利（淨損）歸屬於非控制權益
7,2025-12-31,2330,OperatingIncome,"564,902,413,000.0",營業利益（損失）
8,2025-12-31,2330,TotalNonoperatingIncomeAndExpense,"27,460,788,000.0",營業外收入及支出
9,2025-12-31,2330,CostOfGoodsSold,"394,103,585,000.0",營業成本


### 4. 月營收(TaiwanStockMonthRevenue)

In [ ]:
def fetch_stock_month_revenue(stock_id,  begin_date= date.today().strftime('%Y-%m-%d'), end_date= date.today().strftime('%Y-%m-%d')):
    params = {
        'dataset': 'TaiwanStockMonthRevenue',
        'data_id': stock_id,
        'start_date': begin_date, #API設定一律撈到最新資料
        'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df

"""
    [TaiwanStockFinancialStatements][schema]
    date: str, # 日期
    stock_id: str, # 股票代碼
    country: str, # 國家
    revenue: int64, # 營收
    revenue_month: int64, # 營收月份
    revenue_year: int64 # 營收年份
"""


'\n    [TaiwanStockFinancialStatements][schema]\n    date: str, # 日期\n    stock_id: str, # 股票代碼\n    country: str, # 國家\n    revenue: int64, # 營收\n    revenue_month: int64, # 營收月份\n    revenue_year: int64 # 營收年份\n'

In [64]:
tmp = fetch_stock_month_revenue("2330","2025-11-01")
tmp

,date,stock_id,country,revenue,revenue_month,revenue_year
0,2025-11-01,2330,Taiwan,367473051000,10,2025
1,2025-12-01,2330,Taiwan,343613802000,11,2025
2,2026-01-01,2330,Taiwan,335003568000,12,2025
3,2026-02-01,2330,Taiwan,401255128000,1,2026
4,2026-03-01,2330,Taiwan,317656613000,2,2026


### 5. 資產負債表(TaiwanStockBalanceSheet)

In [ ]:
def fetch_stock_balancesheet(stock_id,  begin_date= date.today().strftime('%Y-%m-%d')):
    params = {
        'dataset': 'TaiwanStockBalanceSheet',
        'data_id': stock_id,
        'start_date': begin_date, #API設定一律撈到最新資料
        'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df
"""
    [TaiwanStockBalanceSheet][schema]
    date: str, # 日期
    stock_id: str, # 股票代碼
    type: str, # 類別
    value: float64, # 數值
    origin_name: str # 原始名稱
"""

"""
重要或是對股價有利的財報項目：
1. 存貨(Inventory)：公司做好了但還沒賣出去的產品，或是還沒製作完成的半成品與原料。
2. 不動產、廠房及設備(Property, Plant, and Equipment, PP&E)：公司買來用於營業、預期使用超過一年的有形資產。
3. 歸屬於母公司業主之權益合計：把公司所有資產變現，並還清所有負債後，真正屬於這家母公司股東的「淨值」。
4. 保留盈餘合計 (Retained Earnings)：公司過去歷年來賺的錢，扣除掉已經發給股東的股息後，選擇「保留」在公司內部的資金總和。
5. 現金及約當現金 (Cash and Cash Equivalents)：公司帳上隨時可以動用的現金，以及三個月內可以輕易變現且風險極低的短期投資（例如定存、國庫券）。
6. 資產總額 (Total Assets)：公司擁有的全部資產的總和，包含流動資產和非流動資產。
"""

'\n    [TaiwanStockBalanceSheet][schema]\n    date: str, # 日期\n    stock_id: str, # 股票代碼\n    type: str, # 類別\n    value: float64, # 數值\n    origin_name: str # 原始名稱\n'

In [86]:
tmp = fetch_stock_balancesheet("2330","2005-12-31")
tmp

,date,stock_id,type,value,origin_name
0,2012-03-31,2330,AccountsPayable_per,1.6,應付帳款
1,2012-03-31,2330,AccountsPayableToRelatedParties,"906,317,000.0",應付帳款－關係人
2,2012-03-31,2330,AccountsPayableToRelatedParties_per,0.1,應付帳款－關係人
3,2012-03-31,2330,AccountsReceivableDuefromRelatedPartiesNet,"647,314,000.0",應收帳款－關係人淨額
4,2012-03-31,2330,AccountsReceivableDuefromRelatedPartiesNet_per,0.1,應收帳款－關係人淨額
...,...,...,...,...,...
5639,2025-12-31,2330,Equity,"5,460,795,283,000.0",權益總額
5640,2025-12-31,2330,Equity_per,68.8,權益總額
5641,2025-12-31,2330,TotalLiabilitiesEquity,"7,933,023,878,000.0",負債及權益總計
5642,2025-12-31,2330,TotalLiabilitiesEquity_per,100.0,負債及權益總計


### 6. 現金流量(TaiwanStockCashFlowsStatement)

In [ ]:
def fetch_stock_cashflow(stock_id,  begin_date= date.today().strftime('%Y-%m-%d')):
    params = {
        'dataset': 'TaiwanStockCashFlowsStatement',
        'data_id': stock_id,
        'start_date': begin_date, #API設定一律撈到最新資料
        'token': TOKEN
    }
    res = requests.get(BASE_URL, params=params).json()
    df = pd.DataFrame(res['data'])
    return df
"""
    [TaiwanStockCashFlowsStatement][schema]
    date: str, # 日期
    stock_id: str, # 股票代碼
    type: str, # 類別
    value: float64, # 數值
    origin_name: str # 原始名稱
"""
"""
重要或是對股價有利的財報項目：
1. 營業活動之淨現金流入：它代表公司「靠本業真正收進口袋的現金」。如果損益表上的 EPS 很高，但這個數字卻是「負數」或「遠低於稅後淨利」，這在財務上稱為「盈餘品質不佳」。市場一旦發現公司只是帳面好看、根本收不到錢，股價通常會面臨猛烈修正。
2. 取得不動產、廠房及設備：這個項目在財務上就是所謂的資本支出，負得越多，市場有時反而越興奮。像台積電這種重資本投資的產業，當這個數字大幅擴張，代表高層對未來的訂單極度樂觀，正在大舉擴產。這往往是推升長線股價的最強燃料。。
3. 應收帳款（增加）減少 & 存貨（增加）減少：這兩個數字是「營業活動現金流」底下的明細，專門用來抓出公司營運的壞習慣。如果應收帳款增加，代表公司賣出產品後，還沒收到錢；如果存貨增加，代表公司做好了產品，但還沒賣出去。這兩個數字如果是「負數」，代表公司在營運上有「積極放水」的嫌疑，可能是為了衝刺業績而放寬信用條件（讓客戶更晚付款），或是為了衝刺業績而過度生產（做了很多賣不掉的產品）。這些都是財務上的紅旗，市場通常會給予嚴厲的懲罰。
4. 籌資活動之淨現金流入（流出）：可以看出公司缺不缺錢。
4-1. 舉借長期借款：大增（籌資現金流入），代表公司正在向銀行借大錢。這時你要去比對前面的 取得不動產、廠房及設備，如果借錢是為了擴廠，那是好事；如果借錢只是為了發放薪水或硬湊錢來發股利，那股價後續通常會崩盤。
4-2. 償還長期借款：如果一家公司長期在 償還長期借款，且籌資活動淨現金是穩定流出的，代表公司賺的錢不僅夠用，還能不斷還債，這種財務健康的體質會獲得價值投資人的青睞。
"""

'\n    [TaiwanStockCashFlowsStatement][schema]\n    date: str, # 日期\n    stock_id: str, # 股票代碼\n    type: str, # 類別\n    value: float64, # 數值\n    origin_name: str # 原始名稱\n'

In [101]:
tmp = fetch_stock_cashflow("2330","2025-12-31")
tmp

,date,stock_id,type,value,origin_name
0,2025-12-31,2330,CashProvidedByInvestingActivities,"-1,144,393,407,000.0",投資活動之淨現金流入（流出）
1,2025-12-31,2330,OtherInvestingActivities,"70,555,692,000.0",其他投資活動
2,2025-12-31,2330,CashBalancesBeginningOfPeriod,"2,127,627,043,000.0",期初現金及約當現金餘額
3,2025-12-31,2330,PropertyAndPlantAndEquipment,"-1,272,410,529,000.0",取得不動產、廠房及設備
4,2025-12-31,2330,HedgingFinancialLiabilities,"566,873,000.0",除列避險之金融負債
5,2025-12-31,2330,CashBalancesIncrease,"640,229,359,000.0",本期現金及約當現金增加（減少）數
6,2025-12-31,2330,RedemptionOfBonds,"-54,310,000,000.0",償還公司債
7,2025-12-31,2330,CashBalancesEndOfPeriod,"2,767,856,402,000.0",期末現金及約當現金餘額
8,2025-12-31,2330,IncomeBeforeIncomeTaxFromContinuingOperations,"2,041,662,840,000.0",繼續營業單位稅前淨利（淨損）
9,2025-12-31,2330,RentalPrincipalRepayments,"-3,496,528,000.0",租賃本金償還


In [102]:
tmp.origin_name.unique()

<StringArray>
[  '投資活動之淨現金流入（流出）',           '其他投資活動',      '期初現金及約當現金餘額',
      '取得不動產、廠房及設備',        '除列避險之金融負債', '本期現金及約當現金增加（減少）數',
            '償還公司債',      '期末現金及約當現金餘額',   '繼續營業單位稅前淨利（淨損）',
           '租賃本金償還',       '本期稅前淨利（淨損）',   '營業活動之淨現金流入（流出）',
             '折舊費用',             '攤銷費用',             '應付帳款',
       '營業活動之淨現金流入',          '存出保證金減少',             '利息費用',
         '收益費損項目合計',       '應收帳款（增加）減少',         '存貨（增加）減少',
    '營運產生之現金流入（流出）',   '籌資活動之淨現金流入（流出）',           '舉借長期借款',
           '償還長期借款',            '支付之利息',             '利息收入']
Length: 27, dtype: str